# 11 squares in a square, by rounding the corners away

The packing is held **overjammed the whole way** — every square squeezed, every contact loaded — and
the **corner radius** is what absorbs the overlap. The particles start as disks and end as sharp
squares; the run starts at a density *above* the one it has to finish at, and only at the very end is
the packing released.

**Why the corner radius and not an edge bulge.** A continuation only helps if the transient degree of
freedom is *conjugate* to the frustration it has to relieve. The frustration here is not local shape,
it is **orientational commensuration**: whether a square sits at 0 or 45 degrees, and whether rows
register with the grid the walls impose. Morphing an octagon into a square through an edge sagitta
cannot touch that — for side 1/2 and alternating diagonal `1 - eps` the bulge is `h ~ sqrt(eps/2)`, and
a `sqrt(eps)` bulge **cannot mediate a finite rotation**. It tracks whichever branch the initial
packing landed in, so the discrete state is frozen in before the ramp begins.

Corner radius is conjugate to it. At `roundness = 1/2` the particle is a disk: no torque, no
orientational frustration, free exploration. Torques then grow **continuously from zero**, and the
`roundness -> 0` limit *requires* four-fold corner meetings, so the compression itself does the
alignment work.

It also fixes a contact pathology. Facet-facet contact at penetration `d` over length `l` costs
`l d^3`; a sharp 90-degree vertex driven into a facet costs `d^4`. The ratio `l/d` **diverges** as
`d -> 0`, so a sharp system under compression relieves pressure by tilting out of alignment — the
opposite of what a dense square packing needs. A corner of radius `r` caps that at scale `r`.

**What is approximate.** The rounding is chorded, not an exact arc: exact rounding under this contact
law needs `(d + r)^3` integrated along a circular arc, whose arc-versus-vertex case is *elliptic*. The
chords keep the whole verified pipeline unchanged. **The endpoint is not approximate** — at
`roundness = 0` the loop is four vertices, an exact square — so the faceting perturbs the path, never
the answer.

Run this from the repo root.

In [ ]:
import os
import sys

# The modules live at the repo root, but a Jupyter kernel starts in the notebook's OWN directory, so
# walk up until model.py is in sight. Works whether this is opened from tests/ or from the root.
root = os.getcwd()
while not os.path.isfile(os.path.join(root, "model.py")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
if root not in sys.path:
    sys.path.insert(0, root)

import time

import numpy as np
from matplotlib import pyplot as plt

import polyContactSystem as pcs
import records
from model import Model
from packing import Packing

## 1. Parameters

`excess` is the one that matters. It is the pair contact energy in units of one particle indented by a
whole side — Cam's `getExcessEnergy` scale, with the particle side `sqrt(area) = lambda` in place of
the mean edge, which agree exactly at `roundness = 0`. Held fixed, it means the packing carries the
same load at every roundness.

In [ ]:
N = 11                        # squares
arcSegments = 6               # chords per corner. 4*(arcSegments+1) vertices; the CUDA kernel caps at 64
excess = 5e-6                 # OVERJAMMED, held all the way. Watch dMax/rIn: 2e-5 already reaches 0.26
steps = 20                    # roundness ramp sub-steps. Never move the shape in one jump
phi0 = 0.45                   # seed density, loose enough that the lattice does not overlap
bias0 = 0.0                   # cos(4 theta) orientation bias; +ve favours 45 deg, -ve favours 0 deg
seed = 0

# Wound CLOCKWISE, so the confining region is its EXTERIOR. The depth law reads membership from the
# winding, and a counter-clockwise wall inverts confinement into an attractive well.
WALL = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 1.0], [1.0, 0.0]])

print(records.describe(N))
print(f"which is phi = {records.maximumDensity(N):.6f} -- the number to get above")

## 2. The rounded square, at fixed area

One parameter, `roundness = r/a`, runs from 1/2 (a disk) to 0 (a sharp square). **Area is held at
exactly 1**, so the packing fraction means the same thing at every roundness and the ramp never
smuggles in a density change.

Both endpoints produce coincident vertices — at 0 the four arc points collapse onto the corner, at 1/2
the straight sides vanish — and a zero-length edge has no tangent, which is exactly what the contact
law divides by. So the loop is filtered on edge length, the vertex count varies along the path, and
the state carried through the ramp is `(x, y, theta)` per body plus a scale, never vertices.

In [ ]:
def shoelace(loop):
    following = np.roll(loop, -1, axis = 0)
    return 0.5 * float(np.sum(loop[:, 0] * following[:, 1] - following[:, 0] * loop[:, 1]))

def roundedSquare(roundness, segments = arcSegments, tolerance = 1e-9):
    """Rounded square of OUTER side 1 and corner radius `roundness`, counter-clockwise."""
    half, inner = 0.5, 0.5 - roundness
    if roundness <= tolerance:
        return np.array([[half, -half], [half, half], [-half, half], [-half, -half]])
    centers = np.array([[inner, inner], [-inner, inner], [-inner, -inner], [inner, -inner]])
    corners = []
    for center, start in zip(centers, np.array([0.0, 0.5, 1.0, 1.5]) * np.pi):
        angles = start + np.linspace(0.0, 0.5 * np.pi, segments + 1)
        corners.append(center + roundness * np.stack([np.cos(angles), np.sin(angles)], axis = 1))
    loop = np.concatenate(corners, axis = 0)
    edges = loop - np.roll(loop, 1, axis = 0)
    return loop[np.hypot(edges[:, 0], edges[:, 1]) > tolerance]

def unitAreaLoop(roundness, segments = arcSegments):
    """The same shape rescaled to area EXACTLY 1 -- normalized against the DISCRETIZED area, not the
    analytic 1 - (4 - pi) r^2, or the discretization error would move the real area along the path."""
    loop = roundedSquare(roundness, segments)
    return loop / np.sqrt(shoelace(loop))

show = [0.5, 0.35, 0.25, 0.15, 0.05, 0.0]
figure, axes = plt.subplots(1, len(show), figsize = (13, 2.3))
for axis, roundness in zip(axes, show):
    loop = unitAreaLoop(roundness)
    closed = np.vstack([loop, loop[:1]])
    axis.fill(closed[:, 0], closed[:, 1], color = "#9ec7ff")
    axis.plot(closed[:, 0], closed[:, 1], color = "#2a78d6", lw = 1.5)
    axis.set_title(f"r = {roundness:.2f}, {len(loop)} vertices", fontsize = 9)
    axis.set_aspect("equal"); axis.axis("off")
    axis.set_xlim(-0.75, 0.75); axis.set_ylim(-0.75, 0.75)
plt.show()

## 3. Rigid bodies: three degrees of freedom each, not two per vertex

The shapes are given and the question is whether they **fit**, so letting vertices move independently
answers a different question. The state is `[x, y, theta] * N + [log lambda]` — 34 numbers for eleven
squares against 616 vertex coordinates — and nothing can drift, fold, or need a constraint.

The chain rule from the vertex gradient `g`, with `x = c + lambda R(theta) u`:

    dE/dc      = sum g                  force
    dE/dtheta  = sum g . J (x - c)      torque about the body's own centre
    dE/dlog(l) = sum g . (x - c)        virial

`setRoundness` rebuilds the packing because the vertex count changes along the path; the state does
not care, which is the whole reason for parametrizing it this way.

In [ ]:
class RigidPacking:
    """N rigid rounded squares in a fixed unit wall, plus one global scale."""

    def __init__(self, count, roundness, wallStiffness = 10.0):
        self.count, self.wallStiffness = count, wallStiffness
        self.setRoundness(roundness)

    def setRoundness(self, roundness):
        self.roundness = roundness
        self.unit = unitAreaLoop(roundness)
        loops = [self.unit + [2.0 * i, 0.0] for i in range(self.count)]
        starts = np.cumsum([0] + [len(l) for l in loops])
        packing = Packing(positions = np.concatenate(loops).reshape(-1), startIndices = starts,
                          box = None, targetArea = 1.0, targetEdgeLength = 1.0)
        model = Model(N = self.count, n = len(self.unit), seed = 0)
        model.packing = packing
        model.addShape(WALL)
        model.pinVertices(np.arange(model.getNumVertices())[-4:])
        model.setBoundaryConditions("fixed")
        # The bodies are rigid, so every shape spring is a constant and its force identically zero.
        model.setSpringConstants(area = 0.0, edge = 0.0, perimeter = 0.0)
        model.setDepthContact(stiffness = 1.0, wallStiffness = self.wallStiffness)
        self.model, self.starts = model, np.asarray(packing.startIndices)

    def place(self, state):
        blocks, scale = state[:-1].reshape(-1, 3), np.exp(state[-1])
        vertices = self.model.packing.positions.reshape(-1, 2)
        for i in range(self.count):
            cosine, sine = np.cos(blocks[i, 2]), np.sin(blocks[i, 2])
            turned = self.unit @ np.array([[cosine, sine], [-sine, cosine]])
            vertices[self.starts[i]:self.starts[i + 1]] = scale * turned + blocks[i, 0:2]
        self.model.packing._forces = self.model._forces = self.model._energy = None

    def energyGradient(self, state, bias = 0.0):
        self.place(state)
        energy, force = pcs.packingEnergyForce(self.model.packing, 1.0,
                                               wallStiffness = self.wallStiffness)
        gradient, vertices = -force, self.model.packing.positions.reshape(-1, 2)
        blocks, out = state[:-1].reshape(-1, 3), np.zeros(3 * self.count + 1)
        rigid = out[:-1].reshape(-1, 3)
        for i in range(self.count):
            rows = gradient[self.starts[i]:self.starts[i + 1]]
            offset = vertices[self.starts[i]:self.starts[i + 1]] - blocks[i, 0:2]
            rigid[i, 0:2] = rows.sum(axis = 0)
            rigid[i, 2] = float(np.sum(rows[:, 1] * offset[:, 0] - rows[:, 0] * offset[:, 1]))
            out[-1] += float(np.sum(rows * offset))
        if bias != 0.0:
            energy += bias * float(np.sum(np.cos(4.0 * blocks[:, 2])))
            rigid[:, 2] -= 4.0 * bias * np.sin(4.0 * blocks[:, 2])
        return energy, out

    def relax(self, state, bias = 0.0, maxSteps = 1200, gradientTolerance = 1e-11):
        """L-BFGS over the ARRANGEMENT at fixed size -- the scale is the controller's to move."""
        from scipy.optimize import minimize
        held = state[-1]
        def objective(reduced):
            energy, gradient = self.energyGradient(np.concatenate([reduced, [held]]), bias)
            return energy, gradient[:-1]
        result = minimize(objective, state[:-1], jac = True, method = "L-BFGS-B",
                          options = {"maxiter": maxSteps, "ftol": 0.0,
                                     "gtol": gradientTolerance, "maxcor": 20})
        out = np.concatenate([result.x, [held]])
        self.place(out)
        return out

    def packingFraction(self, state):  return self.count * np.exp(2.0 * state[-1])
    def side(self, state):             return np.exp(-state[-1])
    def overlapArea(self):             return float(self.model.getOverlapArea())
    def orientations(self, state):     return np.degrees(state[:-1].reshape(-1, 3)[:, 2]) % 90.0

    def validity(self):
        """dMax / rIn, the depth law's ONE hard limit. Past the medial-axis ridge the repulsion
        REVERSES SIGN and bodies are pulled through, so anything near 1 invalidates the run."""
        bodies = pcs.BodySet.__new__(pcs.BodySet)
        bodies.positions = self.model.packing.positions.reshape(-1, 2)
        bodies.startIndices, bodies.boxSize, bodies.exterior = self.starts, None, self.count
        return float(pcs.systemValidity(bodies)[0])

## 4. Excess energy and the two-sided hold

`holdExcess` is the rigid-coordinate version of `Model.holdExcessEnergy`: it **compresses a loose
packing and decompresses an overjammed one** until the relaxed pair contact energy sits at the target,
so the run never has to know its own jamming density in advance.

The wall term is subtracted, for the reason `getPairContactEnergy` records: contact and wall
penetration are **alternatives**, not independent terms, and a criterion written on the total accepts
a packing that is merely leaking out of its container.

The response is steep — `d log(excess) / d log(lambda)` runs about 100 — so the step is a secant in
log-log with a cap rather than a fixed increment.

In [ ]:
def excessEnergy(system, state):
    """Pair contact energy per particle, in units of one particle indented by a whole side."""
    system.place(state)
    total, _ = pcs.packingEnergyForce(system.model.packing, 1.0,
                                      wallStiffness = system.wallStiffness)
    wall, _ = pcs.confinementEnergyGradient(system.model.packing, system.wallStiffness)
    return (total - wall) / (system.count * np.exp(4.0 * state[-1]))

def holdExcess(system, state, target, tolerance = 0.08, maxRounds = 25, maxScaleStep = 1.02):
    state = system.relax(state)
    previous = None
    for _ in range(maxRounds):
        got = excessEnergy(system, state)
        if got > 0.0 and abs(got - target) <= tolerance * target:
            break
        slope = 100.0
        if previous is not None and got > 0.0:
            dScale, dExcess = state[-1] - previous[0], np.log(got) - np.log(previous[1])
            if abs(dScale) > 1e-12 and abs(dExcess) > 1e-6:
                slope = float(np.clip(dExcess / dScale, 20.0, 600.0))
        previous = (state[-1], max(got, 1e-30))
        wanted = np.log(target) - np.log(max(got, 1e-30))
        state = state.copy()
        state[-1] += float(np.clip(wanted / slope, -np.log(maxScaleStep), np.log(maxScaleStep)))
        state = system.relax(state)
    return state, excessEnergy(system, state)

def certify(system, state, rounds = 22):
    """Largest scale at which the arrangement relaxes to EXACTLY zero overlap.

    This is what makes the number an answer rather than an estimate -- everything before it runs at a
    finite excess and so at a finite overlap. It needs NO tolerance: the contact energy is purely
    repulsive, its only zero-energy states are disjoint ones, and the overlap area reads exactly
    0.000e+00 at a feasible scale. It is a LOCAL statement -- an infeasible verdict means THIS
    arrangement does not fit, never that none does."""
    high = low = np.exp(state[-1])
    best = None
    for _ in range(60):
        trial = state.copy(); trial[-1] = np.log(low)
        trial = system.relax(trial, maxSteps = 4000, gradientTolerance = 1e-13)
        if system.overlapArea() == 0.0:
            best = trial.copy(); break
        low *= 0.99
    for _ in range(rounds):
        middle = 0.5 * (low + high)
        trial = state.copy(); trial[-1] = np.log(middle)
        trial = system.relax(trial, maxSteps = 4000, gradientTolerance = 1e-13)
        if system.overlapArea() == 0.0:
            low, best = middle, trial.copy()
        else:
            high = middle
    system.place(best)
    return best

## 5. Start overjammed — above the density we have to finish at

This is the part the first version got wrong. Disks jam near `phi = 0.7007`, and the square answer is
`phi = 0.7318`, so a protocol that tracks *jamming* from the disk end **starts below the density it
has to reach** and spends the ramp trying to climb. Holding the excess instead puts the disk stage at
`phi ~ 0.758`, already above the target, and the density is then free to rise as the corners sharpen.

The centres go on a lattice, not at random: the depth contact energy is **non-monotonic past half
overlap**, so a fully stacked pair is a genuine force-balanced minimum that relaxation walks *into*.
A lattice starts well short of that barrier.

In [ ]:
rng = np.random.default_rng(seed)
columns = int(np.ceil(np.sqrt(N)))
rows = int(np.ceil(N / columns))
gridX, gridY = np.meshgrid(np.arange(columns), np.arange(rows))
cells = np.stack([gridX.ravel(), gridY.ravel()], axis = 1)[:N]

state = np.zeros(3 * N + 1)
state[:-1].reshape(-1, 3)[:, 0] = (cells[:, 0] + 0.5) / columns
state[:-1].reshape(-1, 3)[:, 1] = (cells[:, 1] + 0.5) / rows
state[:-1].reshape(-1, 3)[:, 2] = rng.uniform(0.0, 0.5 * np.pi, N)
state[-1] = 0.5 * np.log(phi0 / N)

system = RigidPacking(N, 0.5)
state, got = holdExcess(system, state, excess, maxRounds = 60)
print(f"disks held at excess {got:.2e}:  phi {system.packingFraction(state):.5f}   "
      f"dMax/rIn {system.validity():.3f}")
print(f"the square record is phi {records.maximumDensity(N):.5f} -- we start ABOVE it")

## 6. The ramp

Per step: set the new corner radius, relax the arrangement, then re-establish the excess. The density
is an **output** of that hold, never a control parameter — a bisection on density teleports the
configuration and re-relaxes, and this landscape is glassy enough that the teleport decides the answer.

Watch `dMax/rIn`. At `excess = 5e-6` it ends near 0.19; at `2e-5` it reaches 0.26, and the law wants it
far below 1.

In [ ]:
history = []
start = time.time()
schedule = 0.5 * (1.0 - np.arange(steps + 1) / steps) ** 1.5

for index, roundness in enumerate(schedule):
    system.setRoundness(roundness)
    bias = bias0 * max(0.0, 1.0 - index / (0.6 * steps))
    if bias != 0.0:
        state = system.relax(state, bias = bias)
    state, got = holdExcess(system, state, excess)
    history.append({"roundness": roundness, "phi": system.packingFraction(state),
                    "excess": got, "validity": system.validity()})
    print(f"  r {roundness:.4f}   phi {system.packingFraction(state):.5f}   "
          f"excess {got:.2e}   dMax/rIn {system.validity():.3f}")
print(f"\nramp took {time.time() - start:.0f}s")

## 7. Release

Walk the excess down by decades, then certify. Only the certified row is a packing — everything above
it carries overlap, so its density is an overestimate.

In [ ]:
for level in (1e-6, 1e-7, 1e-8, 1e-9):
    state, got = holdExcess(system, state, level, maxRounds = 40)
    print(f"  released to excess {got:.2e}   phi {system.packingFraction(state):.6f}   "
          f"side {system.side(state):.6f}")

state = certify(system, state)
print(f"\n{records.describe(N, system.side(state))}")
print(f"overlap area (exact, and independent of the energy): {system.overlapArea():.3e}")
print(f"orientations, deg mod 90: {np.round(np.sort(system.orientations(state)), 1)}")

In [ ]:
figure, axes = plt.subplots(1, 4, figsize = (18, 3.8))
x = [h["roundness"] for h in history]

axes[0].plot(x, [h["phi"] for h in history], marker = "o", color = "#2a78d6")
axes[0].axhline(records.maximumDensity(N), ls = "--", color = "0.5", label = "best known")
axes[0].plot([0.0], [system.packingFraction(state)], "*", color = "#eb6834", ms = 16,
             label = "released + certified")
axes[0].set_ylabel("packing fraction"); axes[0].legend(fontsize = 8)

axes[1].plot(x, [h["excess"] for h in history], marker = "o", color = "#2a78d6")
axes[1].axhline(excess, ls = "--", color = "0.5")
axes[1].set_yscale("log"); axes[1].set_ylabel("excess energy (held)")

axes[2].plot(x, [h["validity"] for h in history], marker = "o", color = "#2a78d6")
axes[2].set_ylabel("dMax / rIn   (the law needs << 1)")

for axis in axes[:3]:
    axis.set_xlabel("roundness  r/a   (disk -> sharp)"); axis.invert_xaxis(); axis.grid(alpha = 0.3)

system.model.draw(ax = axes[3])
axes[3].set_title(f"s = {system.side(state):.5f}", fontsize = 10)
plt.tight_layout(); plt.show()

## 8. Branches

The walls impose 0 degrees, but many record configurations have a **45-degree core** (n = 5, 10, 11,
17, 18 ...), and those are isolated basins no smooth protocol tunnels into. `bias0` adds
`bias * sum cos(4 theta)` — the lowest harmonic compatible with a square's four-fold symmetry —
annealed to zero over the first 60% of the ramp. Run the two signs as **separate populations**, not as
samples of one.

This is a heuristic and certifies nothing. The scatter across seeds is real; run several.

In [ ]:
def run(seed, bias0, excess = excess, steps = steps):
    rng = np.random.default_rng(seed)
    state = np.zeros(3 * N + 1)
    blocks = state[:-1].reshape(-1, 3)
    blocks[:, 0] = (cells[:, 0] + 0.5) / columns
    blocks[:, 1] = (cells[:, 1] + 0.5) / rows
    blocks[:, 2] = rng.uniform(0.0, 0.5 * np.pi, N)
    state[-1] = 0.5 * np.log(phi0 / N)
    system = RigidPacking(N, 0.5)
    state, _ = holdExcess(system, state, excess, maxRounds = 60)
    for index, roundness in enumerate(0.5 * (1.0 - np.arange(steps + 1) / steps) ** 1.5):
        system.setRoundness(roundness)
        bias = bias0 * max(0.0, 1.0 - index / (0.6 * steps))
        if bias != 0.0:
            state = system.relax(state, bias = bias)
        state, _ = holdExcess(system, state, excess)
    for level in (1e-6, 1e-7, 1e-8, 1e-9):
        state, _ = holdExcess(system, state, level, maxRounds = 40)
    return system, certify(system, state)

results = []
for label, bias in (("none", 0.0), ("tilted", 3e-6), ("aligned", -3e-6)):
    for trial in range(3):
        built, final = run(trial, bias)
        results.append({"branch": label, "seed": trial, "side": built.side(final),
                        "system": built, "state": final})
        print(f"{label:>8}  seed {trial}  s = {built.side(final):.5f}  "
              f"({100 * (built.side(final) / records.bestKnownSide(N) - 1):+.2f}%)")

best = min(results, key = lambda row: row["side"])
print(f"\n{records.describe(N, best['side'])}")
print(f"branch {best['branch']}, seed {best['seed']}")